<a href="https://colab.research.google.com/github/slomi23/ML_fx/blob/main/model_experiment_Prophet_clusters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone "https://github.com/slomi23/ML_fx.git"
!cd ML_fx/

Cloning into 'ML_fx'...
remote: Enumerating objects: 87, done.
remote: Counting objects: 100% (87/87), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 87 (delta 34), reused 28 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (87/87), 20.90 MiB | 3.10 MiB/s, done.
Resolving deltas: 100% (34/34), done.
Updating files: 100% (19/19), done.


In [2]:
import pandas as pd
import numpy as np
import os
import zipfile
import io

PROCCESSED_DATA_DIR = "./ML_fx/data/processed/"
train=pd.read_csv(os.path.join(PROCCESSED_DATA_DIR, "train_prepared.csv"))
print(train.head())

   Store  Dept        Date  Weekly_Sales  IsHoliday  Temperature  Fuel_Price  \
0      1     1  2010-02-05      24924.50          0        42.31       2.572   
1      1     1  2010-02-12      46039.49          1        38.51       2.548   
2      1     1  2010-02-19      41595.55          0        39.93       2.514   
3      1     1  2010-02-26      19403.54          0        46.63       2.561   
4      1     1  2010-03-05      21827.90          0        46.50       2.625   

   MarkDown1  MarkDown2  MarkDown3  ...  Type    Size  sales_lag_52  Year  \
0    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
1    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
2    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
3    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   
4    5347.45      192.0       24.6  ...    20  151315       7998.55  2010   

   month_sin     month_cos   dow_sin   dow_cos  week_sin

In [3]:
!pip install wandb -q
!pip install neuralforecast torch pytorch-lightning
!pip install pytorch-forecasting pandas numpy torch matplotlib


import wandb
import os

# Retrieve the secret from Kaggle Secrets

api_key = "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"
if api_key:
    wandb.login(key=api_key)
else:
    print("Warning: could not log in wandb ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.3/425.3 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: slomi23 (slomi23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


# Let's make couple of models grouped by type and size

In [40]:
import pandas as pd
import numpy as np
from prophet import Prophet
import os
import joblib
from joblib import Parallel, delayed
from joblib import Parallel, delayed
import time
import wandb
train_set=train
train_set['size_bin'] = (train_set['Size'] / 1000).round().astype(int) * 1000

# 2. Create a mapping of Cluster Key -> List of (Store, Dept) tuples
# This object tells us which stores/depts belong to which cluster
cluster_members = {}

for (store_type, size_bin), group_df in train_set.groupby(['Type', 'size_bin']):
    # Get unique Store/Dept combinations in this cluster
    members = group_df[['Store', 'Dept']].drop_duplicates().values.tolist()
    if len(members) >= 5:
        key = f"Type_{store_type}_Size_{size_bin}"
        cluster_members[key] = {
            'Type': store_type,
            'size_bin': size_bin,
            'Members': members # List of [Store, Dept] lists
        }

print(f"Created {len(cluster_members)} clusters.")
#run = wandb.init(project="ML_fx_Prophet_Walmart", name="Prophet_Type_Lag_Bins")
holidays_df = pd.DataFrame({
    'holiday': ['Super Bowl', 'Super Bowl', 'Super Bowl', 'Super Bowl',
                'Labor Day', 'Labor Day', 'Labor Day', 'Labor Day',
                'Thanksgiving', 'Thanksgiving', 'Thanksgiving', 'Thanksgiving',
                'Christmas', 'Christmas', 'Christmas', 'Christmas'],
    'ds': pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08',
                          '2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06',
                          '2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29',
                          '2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])
})

Created 34 clusters.


In [41]:
def fit_prophet_cluster(cluster_key, store_type, size_bin):
  try:
    df = train_set[(train_set['Type'] == store_type) & (train_set['size_bin'] == size_bin)].copy()
    df = df.sort_values('Date').reset_index(drop=True)
    df=df[['Date', 'Weekly_Sales']]
    df = df.groupby('Date')['Weekly_Sales'].mean().reset_index()
    print(df.head())

    val_start_date = '2011-12-01'
    train_df = df[df['Date'] <= val_start_date]
    val_df = df[df['Date'] > val_start_date]
    train_df=train_df[['Date', 'Weekly_Sales']]
    val_df=val_df[['Date', 'Weekly_Sales']]

    prophet_train = train_df.rename(columns={'Date': 'ds', 'Weekly_Sales': 'y'})
    prophet_val = val_df.rename(columns={'Date': 'ds'})
    model = Prophet(daily_seasonality=False, weekly_seasonality=True, yearly_seasonality=True)
    model.fit(prophet_train)
    forecast = model.predict(prophet_val)
    val_actuals = val_df['Weekly_Sales'].values
    val_preds = forecast['yhat'].values
    mae = np.mean(np.abs(val_actuals - val_preds))
    print(mae)
    return {'model': model, 'mae': mae}
  except Exception as e:
      print(f"Error fitting cluster {cluster_key}: {e}")
      return None
fit_prophet_cluster(1, 20, 151000)

         Date  Weekly_Sales
0  2010-02-05  22516.313699
1  2010-02-12  22804.964444
2  2010-02-19  22081.755753
3  2010-02-26  19579.549861
4  2010-03-05  21298.721644
1157.7483376701566


{'model': <prophet.forecaster.Prophet at 0x799cba01e9f0>,
 'mae': np.float64(1157.7483376701566)}

In [42]:

run = wandb.init(project="ML_fx_Prophet_Walmart", name="Prophet_Type_Size_Bins")
# Create directory for models
MODEL_DIR = './ML_fx/prophet_cluster_models'
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Training {len(cluster_members)} cluster models...")

def train_and_save_cluster(idx, key, info):
    try:
        result = fit_prophet_cluster(key, info['Type'], info['size_bin'])
        if result is not None:
            model = result['model']
            mae = result['mae']

            filepath = os.path.join(MODEL_DIR, f"{key}.pkl")
            joblib.dump(model, filepath)

            # Return success status WITH MAE
            return {'status': 'success', 'key': key, 'mae': mae}
        else:
            return {'status': 'failed', 'key': key, 'reason': 'No data or error'}
    except Exception as e:
        return {'status': 'error', 'key': key, 'reason': str(e)}

# Run in parallel
start_time = time.time()
results = Parallel(n_jobs=-1)(
    delayed(train_and_save_cluster)(i, key, info)
    for i, (key, info) in enumerate(cluster_members.items())
)

end_time = time.time()
print(f"Finished training and saving in {end_time - start_time:.2f} seconds")

# Count successes
successes = [r for r in results if r['status'] == 'success']
failures = [r for r in results if r['status'] != 'success']

print(f"Successfully saved {len(successes)} models.")
if successes:
    maes = [r['mae'] for r in successes]
    avg_mae = np.mean(maes)
    # Log metrics to W&B
    for mae in maes:
      print(mae)
    run.log({
        "average_cluster_mae": avg_mae,

        "num_clusters_trained": len(successes),
        "min_cluster_mae": min(maes),
        "max_cluster_mae": max(maes)
    })

    # Log histogram of MAEs across clusters
    run.log({"cluster_mae_distribution": wandb.Histogram(maes)})

    print(f"Average Cluster MAE: {avg_mae:.2f}")
if failures:
    print(f"Failed to save {len(failures)} models.")
    for f in failures[:5]: # Print first 5 errors
        print(f"  - {f['key']}: {f.get('reason', 'Unknown')}")

run.finish()

Training 34 cluster models...
Finished training and saving in 29.12 seconds
Successfully saved 34 models.
342.9521690181423
1094.5411740211287
1130.8359095272765
364.59959121463197
383.8706702020386
655.4753323927536
1035.4006857318723
694.4121293030927
632.1733447321905
2171.3814290582322
787.3823097250366
1304.6094757125275
680.8822751338623
914.3248115589913
2141.9947983732072
672.4718220785221
559.4391783817213
1666.1908689067943
671.0000814319202
1376.2585105424116
358.1648693623577
1157.7483376701566
665.1833850784419
681.4064296729621
683.7723334923089
1074.4867028549913
1059.2731859183002
2725.6755460377813
2231.351885016989
987.6680214048644
1091.4075876861205
1640.7922302117192
1082.1800381083722
1584.2598892360631
Average Cluster MAE: 1067.75


average_cluster_mae,▁
max_cluster_mae,▁
min_cluster_mae,▁
num_clusters_trained,▁
average_cluster_mae,1067.75197
max_cluster_mae,2725.67555
min_cluster_mae,342.95217
num_clusters_trained,34


In [43]:
from google.colab import files
import shutil

# Zip the folder (adjust path if needed, assuming it's in current dir)
shutil.make_archive('prophet_cluster_models', 'zip', './ML_fx', 'prophet_cluster_models')

# Download
files.download('prophet_cluster_models.zip')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>